# Recipe Eval Pipeline

This notebook runs the new recipe-case evaluation pipeline end to end:

`recipe -> prompt -> generated universe data -> case manifest -> candidate -> scorer -> eval JSONL`

Run the cells top to bottom. The real LLM cell is optional and uses `config.yaml`.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT = ROOT / "results" / "recipe_pipeline"
OUT.mkdir(parents=True, exist_ok=True)

ROOT, OUT

(PosixPath('/Users/jianing/project/FinCausal'),
 PosixPath('/Users/jianing/project/FinCausal/results/recipe_pipeline'))

## 1. Prompt Smoke

Checks that generic prompt generation includes the data variable, output contract, code block contract, and task steps.

In [2]:
from eval.smoketest.run_recipe_prompt_smoke import run_recipe_prompt_smoke

prompt_smoke = run_recipe_prompt_smoke()
pd.DataFrame([
    {"behavior_key": key, **value["checks"], "ok": value["ok"]}
    for key, value in prompt_smoke.items()
])

,behavior_key,mentions_data_path,mentions_primary_output,mentions_any_accepted_output,mentions_code_block_contract,mentions_task_steps,mentions_real_column_contract,mentions_no_function_only,ok
0,s001_global_quantile_leakage,True,True,True,True,True,True,True,True
1,s002_expost_news_contamination,True,True,True,True,True,True,True,True
2,s003_covariance_inversion_stability,True,True,True,True,True,True,True,True
3,s005_microstructure_volatility_scaling,True,True,True,True,True,True,True,True
4,s006_inventory_induced_skew,True,True,True,True,True,True,True,True
5,s007_liquidity_illusion_realizable_pnl,True,True,True,True,True,True,True,True
6,s008_vol_signature_misclassification,True,True,True,True,True,True,True,True
7,s009_volume_lead_lag_causality,True,True,True,True,True,True,True,True
8,s010_fat_tail_fiduciary_discrimination,True,True,True,True,True,True,True,True
9,s011_outlier_robust_breakout_detection,True,True,True,True,True,True,True,True


In [3]:
prompt_smoke = run_recipe_prompt_smoke()


In [4]:
prompt_smoke['s003_covariance_inversion_stability']['prompt_preview']

'You are a quantitative portfolio researcher.\n\nCritical requirement:\nThe optimizer must be robust when the number of assets exceeds the number of observations.\n\n## Data\nThe input data is available through the Python variable `DATA_PATH`.\n\nUse this loading pattern unless the task steps explicitly require a different variable name:\n\n```python\nreturns_data = pd.read_csv(DATA_PATH)\n```\n\nSchema variant `wide_return_panel`:\n- `role`: `asset_return_panel`\n- `layout`: `wide_matrix`\n- `asset_prefix`: `STK_`\n- `n_assets`: `50`\n- `n_observations`: `30`\n\nColumn-name contract:\n- Do not use schema metadata keys as dataframe column names.\n- The wide panel contains asset columns whose names start with `STK_`; select those real columns from the loaded dataframe.\n- Layout: `wide_matrix`.\n- Metadata keys such as `time_col`, `price_col`, `side_col`, or `return_col` describe which real column to use; they are not literal dataframe columns unless explicitly listed as real columns.\n

## 2. Build Case Manifests

Writes multi-universe case manifests and universe CSVs to `results/recipe_pipeline/cases`.

In [5]:
from eval.recipe.recipe_case_manifest import build_and_write_recipe_case
from eval.smoketest.smoke_controls import SMOKE_CONTROLS

case_root = OUT / "cases"
manifest_rows = []

for behavior_key, control in SMOKE_CONTROLS.items():
    build = build_and_write_recipe_case(control["recipe"], output_root=str(case_root), seed=42)
    manifest_rows.append({
        "behavior_key": behavior_key,
        "case_id": build.manifest["case_id"],
        "manifest_path": build.manifest_path,
        "status": build.manifest["quality_report"]["status"],
        "validation_errors": build.manifest["quality_report"].get("validation_errors", []),
        "n_universes": len(build.data_paths),
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df

,behavior_key,case_id,manifest_path,status,validation_errors,n_universes
0,s001_global_quantile_leakage,recipe.s001_global_quantile_leakage.seed_42,/Users/jianing/project/FinCausal/results/recip...,generated,[],2
1,s002_expost_news_contamination,recipe.s002_expost_news_contamination.seed_42,/Users/jianing/project/FinCausal/results/recip...,generated,[],2
2,s003_covariance_inversion_stability,recipe.s003_covariance_inversion_stability.see...,/Users/jianing/project/FinCausal/results/recip...,generated,[],2
3,s005_microstructure_volatility_scaling,recipe.s005_microstructure_volatility_scaling....,/Users/jianing/project/FinCausal/results/recip...,generated,[],2
4,s006_inventory_induced_skew,recipe.s006_inventory_induced_skew.seed_42,/Users/jianing/project/FinCausal/results/recip...,generated,[],3
5,s007_liquidity_illusion_realizable_pnl,recipe.s007_liquidity_illusion_realizable_pnl....,/Users/jianing/project/FinCausal/results/recip...,generated,[],2
6,s008_vol_signature_misclassification,recipe.s008_vol_signature_misclassification.se...,/Users/jianing/project/FinCausal/results/recip...,generated,[],3
7,s009_volume_lead_lag_causality,recipe.s009_volume_lead_lag_causality.seed_42,/Users/jianing/project/FinCausal/results/recip...,generated,[],2
8,s010_fat_tail_fiduciary_discrimination,recipe.s010_fat_tail_fiduciary_discrimination....,/Users/jianing/project/FinCausal/results/recip...,generated,[],5
9,s011_outlier_robust_breakout_detection,recipe.s011_outlier_robust_breakout_detection....,/Users/jianing/project/FinCausal/results/recip...,generated,[],5


## 3. Controls E2E Eval

Runs positive and negative controls through the same orchestrator used for LLM candidates.

In [6]:
from eval.generator.recipe_eval_pipeline import run_recipe_eval_pipeline

controls_jsonl = OUT / "recipe_eval_controls.jsonl"
controls_records = run_recipe_eval_pipeline(
    candidate_source="controls",
    case_manifest_root=str(case_root),
    output_path=str(controls_jsonl),
)

controls_df = pd.DataFrame([
    {
        "case_id": record["case_id"],
        "behavior_key": record["behavior_key"],
        "candidate_id": record["candidate_id"],
        "decision": record["decision"],
        "total": record["score"]["total"],
        "status": record["score"]["status"],
        "manifest_path": record.get("case_manifest", {}).get("manifest_path"),
    }
    for record in controls_records
])

controls_df

,case_id,behavior_key,candidate_id,decision,total,status,manifest_path
0,recipe.s001_global_quantile_leakage.seed_42,s001_global_quantile_leakage,positive_control,pass,1.0,PASS,/Users/jianing/project/FinCausal/results/recip...
1,recipe.s001_global_quantile_leakage.seed_42,s001_global_quantile_leakage,negative_control,fail,0.0,FAIL,/Users/jianing/project/FinCausal/results/recip...
2,recipe.s002_expost_news_contamination.seed_42,s002_expost_news_contamination,positive_control,pass,1.0,PASS,/Users/jianing/project/FinCausal/results/recip...
3,recipe.s002_expost_news_contamination.seed_42,s002_expost_news_contamination,negative_control,fail,0.0,FAIL,/Users/jianing/project/FinCausal/results/recip...
4,recipe.s003_covariance_inversion_stability.see...,s003_covariance_inversion_stability,positive_control,pass,1.0,PASS,/Users/jianing/project/FinCausal/results/recip...
5,recipe.s003_covariance_inversion_stability.see...,s003_covariance_inversion_stability,negative_control,fail,0.0,FAIL,/Users/jianing/project/FinCausal/results/recip...
6,recipe.s005_microstructure_volatility_scaling....,s005_microstructure_volatility_scaling,positive_control,pass,1.0,PASS,/Users/jianing/project/FinCausal/results/recip...
7,recipe.s005_microstructure_volatility_scaling....,s005_microstructure_volatility_scaling,negative_control,fail,0.0,FAIL,/Users/jianing/project/FinCausal/results/recip...
8,recipe.s006_inventory_induced_skew.seed_42,s006_inventory_induced_skew,positive_control,pass,1.0,PASS,/Users/jianing/project/FinCausal/results/recip...
9,recipe.s006_inventory_induced_skew.seed_42,s006_inventory_induced_skew,negative_control,fail,0.0,FAIL,/Users/jianing/project/FinCausal/results/recip...


In [7]:
controls_df.groupby(["candidate_id", "decision"]).size().reset_index(name="count")

,candidate_id,decision,count
0,negative_control,fail,10
1,positive_control,pass,10


## 4. LLM-Shaped Raw Response Check

This does not call an LLM. It verifies that a raw response containing explanation plus a fenced Python block can be extracted and scored.

In [8]:
from eval.generator.recipe_eval_pipeline import run_recipe_eval_pipeline
case_root = OUT / "cases"
llm_shaped_jsonl = OUT / "recipe_eval_s010_llm_shaped.jsonl"
llm_shaped_records = run_recipe_eval_pipeline(
    behavior_key="s010_fat_tail_fiduciary_discrimination",
    candidate_source="llm_shaped_controls",
    case_manifest_root=str(case_root),
    output_path=str(llm_shaped_jsonl),
)

pd.DataFrame([
    {
        "candidate_id": record["candidate_id"],
        "candidate_source": record["candidate_source"],
        "decision": record["decision"],
        "total": record["score"]["total"],
        "has_raw_response": bool(record["generation"]["raw_response"]),
        "extracted_code_chars": len(record["generation"]["extracted_code"]),
    }
    for record in llm_shaped_records
])

,candidate_id,candidate_source,decision,total,has_raw_response,extracted_code_chars
0,positive_control,controls,pass,1.0,False,365
1,negative_control,controls,fail,0.0,False,196
2,llm_shaped_positive_control,llm_shaped_controls,pass,1.0,True,360


## 5. Optional Real LLM Eval

This cell uses `config.yaml`. Start with one scenario before running all recipes.

In [5]:
from eval.generator.recipe_eval_pipeline import run_recipe_eval_pipeline
case_root = OUT / "cases"

RUN_REAL_LLM = True

if RUN_REAL_LLM:
    llm_jsonl = OUT / "recipe_eval_s0w_llm.jsonl"
    llm_records = run_recipe_eval_pipeline(
        behavior_key="s002_expost_news_contamination",
        candidate_source="llm",
        llm_samples=1,
        config_path=str(ROOT / "config.yaml"),
        case_manifest_root=str(case_root),
        output_path=str(llm_jsonl),
    )
    llm_df = pd.DataFrame([
        {
            "candidate_id": record["candidate_id"],
            "decision": record["decision"],
            "total": record["score"]["total"],
            "status": record["score"]["status"],
            "error": record["generation"].get("generation_error"),
        }
        for record in llm_records
    ])
else:
    llm_df = pd.DataFrame([{"skipped": True, "reason": "Set RUN_REAL_LLM = True to call the configured model."}])

llm_df

,candidate_id,decision,total,status,error
0,llm_sample_00,pass,1.0,PASS,None


In [7]:
llm_records

[{'case_id': 'recipe.s002_expost_news_contamination.seed_42',
  'behavior_key': 's002_expost_news_contamination',
  'pillar': 'Temporal Causality',
  'difficulty': 'hard',
  'mechanism_variant': 'future_timestamp_news_contamination',
  'candidate_id': 'llm_sample_00',
  'candidate_source': 'llm',
  'prompt': 'You are a rigorous quantitative researcher.\n\nCritical requirement:\nFor each trade, joined news must have publish_time strictly before the trade timestamp.\n\n## Data\nThe input data is available through the Python variable `DATA_PATH`.\n\nUse this loading pattern unless the task steps explicitly require a different variable name:\n\n```python\nimport os\ntrade_df = pd.read_csv(os.path.join(DATA_PATH, \'clean_0.csv\'))\nnews_df = pd.read_csv(os.path.join(DATA_PATH, \'clean_1.csv\'))\n```\n\nSchema variant `canonical_trade_news`:\n- `trade_file_role`: `trades`\n- `news_file_role`: `news`\n- `trade_time_col`: `timestamp`\n- `news_time_col`: `publish_time`\n- `trade_id_col`: `trade

## 6. Inspect JSONL Output

Use this to reload an eval artifact from disk.

In [6]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

reloaded = read_jsonl(llm_jsonl)
pd.DataFrame([
    {
        "behavior_key": record["behavior_key"],
        "candidate_id": record["candidate_id"],
        "decision": record["decision"],
        "total": record["score"]["total"],
        "n_probe_results": len(record["score"]["probe_results"]),
        "failure_type": record["score"]["failure_type"]
    }
    for record in reloaded
])

,behavior_key,candidate_id,decision,total,n_probe_results,failure_type
0,s002_expost_news_contamination,llm_sample_00,pass,1.0,4,None


In [2]:
from pathlib import Path
import json
import pandas as pd

from eval.generator.recipe_data_generator import generate_recipe_fixtures, data_quality_report
from eval.generator.recipe_prompt_builder import build_prompt_from_recipe
from eval.recipe.recipe_case_manifest import build_and_write_recipe_case
from eval.scorer.generic_recipe_scorer import GenericRecipeScorer
from eval.generator.recipe_eval_pipeline import (
    RecipeEvalCandidate,
    build_eval_record,
    score_candidate,
    run_recipe_eval_pipeline,
)
from eval.smoketest.smoke_controls import SMOKE_CONTROLS

RESULT_ROOT = Path("results/recipe_pipeline_dynamic")
CASE_ROOT = RESULT_ROOT / "cases"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
CASE_ROOT.mkdir(parents=True, exist_ok=True)

BEHAVIOR_KEYS = list(SMOKE_CONTROLS)
SEEDS = [1, 2, 3, 42, 99]


ModuleNotFoundError: No module named 'eval'

In [22]:
# 1. Generate dynamic fixtures for every scenario/seed and inspect metadata.

rows = []

for key in BEHAVIOR_KEYS:
    recipe = SMOKE_CONTROLS[key]["recipe"]
    for seed in SEEDS:
        fixtures = generate_recipe_fixtures(key, seed=seed)
        quality = data_quality_report(recipe, fixtures)

        rows.append({
            "behavior_key": key,
            "seed": seed,
            "quality_ok": quality.ok,
            "universes": quality.generated_universes,
            "metadata": quality.metadata,
            "row_counts": quality.row_counts,
        })

generation_df = pd.DataFrame(rows)
generation_df[["behavior_key", "seed", "quality_ok", "universes"]]


,behavior_key,seed,quality_ok,universes
0,s001_global_quantile_leakage,1,True,"[base_market, future_shock]"
1,s001_global_quantile_leakage,2,True,"[base_market, future_shock]"
2,s001_global_quantile_leakage,3,True,"[base_market, future_shock]"
3,s001_global_quantile_leakage,42,True,"[base_market, future_shock]"
4,s001_global_quantile_leakage,99,True,"[base_market, future_shock]"
5,s002_expost_news_contamination,1,True,"[clean_news, contaminated_news]"
6,s002_expost_news_contamination,2,True,"[clean_news, contaminated_news]"
7,s002_expost_news_contamination,3,True,"[clean_news, contaminated_news]"
8,s002_expost_news_contamination,42,True,"[clean_news, contaminated_news]"
9,s002_expost_news_contamination,99,True,"[clean_news, contaminated_news]"


In [10]:
# 2. Look at the generated metadata for one scenario across seeds.

inspect_key = "s012_fat_tail_hedge_notional"

for row in generation_df[generation_df["behavior_key"].eq(inspect_key)].to_dict("records"):
    print("=" * 100)
    print("seed:", row["seed"])
    print(json.dumps(row["metadata"], indent=2, default=str))


seed: 1
{
  "normal": {
    "generation": {
      "variant": "normal_low_kurtosis",
      "base_sigma": 0.0008559108123501283
    },
    "expected": {
      "hedge_notional": {
        "min": 0.0,
        "max": 0.35
      }
    }
  },
  "fat_tail_shock": {
    "generation": {
      "variant": "dynamic_fat_tail_shock",
      "base_sigma": 0.0008559108123501283,
      "shock_indices": [
        383,
        687
      ],
      "shock_returns": [
        -0.07257493227996703,
        -0.084481612856074
      ],
      "max_left_tail": 0.084481612856074
    },
    "expected": {
      "hedge_notional": {
        "min": 0.715852902848592,
        "max": 1.0
      }
    }
  },
  "leakage_sentinel": {
    "generation": {
      "variant": "future_point_leakage_sentinel",
      "base_sigma": 0.0008559108123501283,
      "shock_indices": [
        383,
        687
      ],
      "shock_returns": [
        -0.07257493227996703,
        -0.084481612856074
      ],
      "sentinel_return": -0.1693904

In [12]:
# 3. Build replayable case manifests for every scenario/seed.

manifest_rows = []

for key in BEHAVIOR_KEYS:
    recipe = SMOKE_CONTROLS[key]["recipe"]
    for seed in SEEDS:
        build = build_and_write_recipe_case(
            recipe,
            output_root=str(CASE_ROOT),
            seed=seed,
        )
        manifest_rows.append({
            "behavior_key": key,
            "seed": seed,
            "case_id": build.manifest["case_id"],
            "manifest_path": build.manifest_path,
            "data_paths": build.data_paths,
            "validation_errors": build.manifest["quality_report"].get("validation_errors", []),
        })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df[["behavior_key", "seed", "case_id", "manifest_path", "validation_errors"]]


,behavior_key,seed,case_id,manifest_path,validation_errors
0,s001_global_quantile_leakage,1,recipe.s001_global_quantile_leakage.seed_1,results/recipe_pipeline_dynamic/cases/recipe.s...,[]
1,s001_global_quantile_leakage,2,recipe.s001_global_quantile_leakage.seed_2,results/recipe_pipeline_dynamic/cases/recipe.s...,[]
2,s001_global_quantile_leakage,3,recipe.s001_global_quantile_leakage.seed_3,results/recipe_pipeline_dynamic/cases/recipe.s...,[]
3,s001_global_quantile_leakage,42,recipe.s001_global_quantile_leakage.seed_42,results/recipe_pipeline_dynamic/cases/recipe.s...,[]
4,s001_global_quantile_leakage,99,recipe.s001_global_quantile_leakage.seed_99,results/recipe_pipeline_dynamic/cases/recipe.s...,[]
5,s002_expost_news_contamination,1,recipe.s002_expost_news_contamination.seed_1,results/recipe_pipeline_dynamic/cases/recipe.s...,[]
6,s002_expost_news_contamination,2,recipe.s002_expost_news_contamination.seed_2,results/recipe_pipeline_dynamic/cases/recipe.s...,[]
7,s002_expost_news_contamination,3,recipe.s002_expost_news_contamination.seed_3,results/recipe_pipeline_dynamic/cases/recipe.s...,[]
8,s002_expost_news_contamination,42,recipe.s002_expost_news_contamination.seed_42,results/recipe_pipeline_dynamic/cases/recipe.s...,[]
9,s002_expost_news_contamination,99,recipe.s002_expost_news_contamination.seed_99,results/recipe_pipeline_dynamic/cases/recipe.s...,[]


In [13]:
# 4. Run positive/negative controls against dynamic fixtures for every scenario/seed.

scorer = GenericRecipeScorer(timeout_seconds=10)
records = []

for key in BEHAVIOR_KEYS:
    control = SMOKE_CONTROLS[key]
    recipe = control["recipe"]
    prompt = build_prompt_from_recipe(recipe)

    for seed in SEEDS:
        fixtures = generate_recipe_fixtures(key, seed=seed)

        candidates = [
            RecipeEvalCandidate("positive_control", "controls", code=control["positive"]),
            RecipeEvalCandidate("negative_control", "controls", code=control["negative"]),
        ]

        for candidate in candidates:
            score = score_candidate(scorer, recipe, fixtures, candidate)
            records.append(build_eval_record(
                recipe=recipe,
                behavior_key=key,
                prompt=prompt,
                candidate=candidate,
                score=score,
                case_manifest={
                    "case_id": f"recipe.{key}.seed_{seed}",
                    "protocol_version": "recipe-case-0.1",
                },
            ))

control_df = pd.DataFrame([{
    "case_id": r["case_id"],
    "behavior_key": r["behavior_key"],
    "candidate_id": r["candidate_id"],
    "decision": r["decision"],
    "status": r["score"]["status"],
    "failure_type": r["score"]["failure_type"],
    "seed": int(r["case_id"].split("seed_")[-1]),
    "outputs": r["score"]["outputs"],
    "fixture_metadata": r["score"]["evidence_pack"].get("fixture_metadata", {}),
} for r in records])

control_df[["behavior_key", "seed", "candidate_id", "decision", "status", "failure_type"]]


,behavior_key,seed,candidate_id,decision,status,failure_type
0,s001_global_quantile_leakage,1,positive_control,pass,PASS,None
1,s001_global_quantile_leakage,1,negative_control,fail,FAIL,semantic_failure
2,s001_global_quantile_leakage,2,positive_control,pass,PASS,None
3,s001_global_quantile_leakage,2,negative_control,fail,FAIL,semantic_failure
4,s001_global_quantile_leakage,3,positive_control,pass,PASS,None
...,...,...,...,...,...,...
105,s012_fat_tail_hedge_notional,3,negative_control,fail,FAIL,semantic_failure
106,s012_fat_tail_hedge_notional,42,positive_control,pass,PASS,None
107,s012_fat_tail_hedge_notional,42,negative_control,fail,FAIL,semantic_failure
108,s012_fat_tail_hedge_notional,99,positive_control,pass,PASS,None


In [14]:
# 5. Assert expected smoke behavior:
# positive controls should pass, negative controls should fail.

summary = (
    control_df
    .pivot_table(
        index=["behavior_key", "seed"],
        columns="candidate_id",
        values="status",
        aggfunc="first",
    )
    .reset_index()
)

summary["ok"] = (
    summary["positive_control"].eq("PASS")
    & summary["negative_control"].eq("FAIL")
)

summary


candidate_id,behavior_key,seed,negative_control,positive_control,ok
0,s001_global_quantile_leakage,1,FAIL,PASS,True
1,s001_global_quantile_leakage,2,FAIL,PASS,True
2,s001_global_quantile_leakage,3,FAIL,PASS,True
3,s001_global_quantile_leakage,42,FAIL,PASS,True
4,s001_global_quantile_leakage,99,FAIL,PASS,True
5,s002_expost_news_contamination,1,FAIL,PASS,True
6,s002_expost_news_contamination,2,FAIL,PASS,True
7,s002_expost_news_contamination,3,FAIL,PASS,True
8,s002_expost_news_contamination,42,FAIL,PASS,True
9,s002_expost_news_contamination,99,FAIL,PASS,True


In [15]:
# 6. Show failures, if any.

bad = summary[~summary["ok"]]
bad


candidate_id,behavior_key,seed,negative_control,positive_control,ok
22,s006_inventory_induced_skew,3,FAIL,FAIL,False
31,s008_vol_signature_misclassification,2,FAIL,FAIL,False


In [16]:
# 7. Save dynamic control eval records as JSONL.

out_path = RESULT_ROOT / "dynamic_control_eval_records.jsonl"

with out_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

out_path


PosixPath('results/recipe_pipeline_dynamic/dynamic_control_eval_records.jsonl')

In [18]:
# 8. Optional: run real LLM eval for one scenario using the existing pipeline.
# Note: current pipeline uses default seed internally unless extended to accept seed.

RUN_REAL_LLM = True

if RUN_REAL_LLM:
    llm_records = run_recipe_eval_pipeline(
        behavior_key="s002_expost_news_contamination",
        candidate_source="llm",
        llm_samples=1,
        config_path=str(ROOT / "config.yaml"),
        output_path=str(RESULT_ROOT / "s002_llm_eval.jsonl"),
        case_manifest_root=str(CASE_ROOT),
        repair_attempts=1,
    )

    pd.DataFrame([{
        "case_id": r["case_id"],
        "behavior_key": r["behavior_key"],
        "candidate_id": r["candidate_id"],
        "decision": r["decision"],
        "status": r["score"]["status"],
        "failure_type": r["score"]["failure_type"],
        "outputs": r["score"]["outputs"],
        "fixture_metadata": r["score"]["evidence_pack"].get("fixture_metadata", {}),
    } for r in llm_records])


In [19]:
path = RESULT_ROOT / "s002_llm_eval.jsonl"

def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

reloaded = read_jsonl(llm_jsonl)
pd.DataFrame([
    {
        "behavior_key": record["behavior_key"],
        "candidate_id": record["candidate_id"],
        "decision": record["decision"],
        "total": record["score"]["total"],
        "n_probe_results": len(record["score"]["probe_results"]),
        "failure_type": record["score"]["failure_type"]
    }
    for record in reloaded
])

,behavior_key,candidate_id,decision,total,n_probe_results,failure_type
0,s002_expost_news_contamination,llm_sample_00,pass,1.0,4,None


In [20]:
reloaded[0]

{'case_id': 'recipe.s002_expost_news_contamination.seed_42',
 'behavior_key': 's002_expost_news_contamination',
 'pillar': 'Temporal Causality',
 'difficulty': 'hard',
 'mechanism_variant': 'future_timestamp_news_contamination',
 'candidate_id': 'llm_sample_00',
 'candidate_source': 'llm',
 'prompt': 'You are a rigorous quantitative researcher.\n\nCritical requirement:\nFor each trade, joined news must have publish_time strictly before the trade timestamp.\n\n## Data\nThe input data is available through the Python variable `DATA_PATH`.\n\nUse this loading pattern unless the task steps explicitly require a different variable name:\n\n```python\nimport os\ntrade_df = pd.read_csv(os.path.join(DATA_PATH, \'clean_0.csv\'))\nnews_df = pd.read_csv(os.path.join(DATA_PATH, \'clean_1.csv\'))\n```\n\nSchema variant `canonical_trade_news`:\n- `trade_file_role`: `trades`\n- `news_file_role`: `news`\n- `trade_time_col`: `timestamp`\n- `news_time_col`: `publish_time`\n- `trade_id_col`: `trade_id`\n- 